# AC-MOT FINAL — DeepSORT SOTA Comparison

**System:** DeepSORT via `deep-sort-realtime` package  
**Detector:** YOLOv8n (same as all other runs — fair comparison)  
**Sequences:** 12 valid VisDrone sequences  
**Est. time:** ~20 min on T4  
**Saves to:** `MyDrive/visdrone/VisDrone_Results/`

In [ ]:
from pathlib import Path

base = Path('/content/drive/MyDrive/visdrone')
for p in sorted(base.rglob('*'))[:30]:
    print(p)

In [ ]:
!pip install ultralytics deep-sort-realtime motmetrics opencv-python-headless pandas numpy tqdm lap -q

import time, shutil, gc
from pathlib import Path
from datetime import datetime
import cv2, numpy as np, pandas as pd, torch, motmetrics as mm
from tqdm import tqdm
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

try: torch.backends.cudnn.benchmark = True
except: pass

DATASET_ROOT  = Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
SEQ_DIR       = DATASET_ROOT / 'sequences'
ANNOT_DIR     = DATASET_ROOT / 'annotations'
DRIVE_RESULTS = Path('/content/drive/MyDrive/visdrone/VisDrone_Results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
LOCAL_TMP     = Path('/content/_tmp')

assert SEQ_DIR.exists(), f'Dataset not found: {SEQ_DIR}'

KNOWN_VALID = [
    'uav0000009_03358_v', 'uav0000073_00600_v', 'uav0000077_00720_v',
    'uav0000088_00290_v', 'uav0000119_02301_v', 'uav0000188_00000_v',
    'uav0000201_00000_v', 'uav0000249_00001_v', 'uav0000249_02688_v',
    'uav0000297_00000_v', 'uav0000306_00230_v', 'uav0000355_00001_v',
]
by_name  = {s.name: s for s in SEQ_DIR.iterdir() if s.is_dir()}
VAL_SEQS = [by_name[n] for n in KNOWN_VALID if n in by_name]

MODEL_NAME = 'yolov8n.pt'
DEVICE     = '0' if torch.cuda.is_available() else 'cpu'
HALF       = DEVICE != 'cpu'

print(f'Device={DEVICE} | FP16={HALF} | Sequences={len(VAL_SEQS)}')
print('System: DeepSORT (deep-sort-realtime) — Drive already mounted')

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────
def load_gt(p):
    df = pd.read_csv(p, header=None,
                     names=['frame','id','x','y','w','h','score','cat','trunc','occ'])
    df = df[df['cat'].isin([1,4,5,6,9])]
    return df[(df['occ']<2) & (df['trunc']<2) & (df['score']==1)].reset_index(drop=True)

def iou_dist(pred, gt):
    if not len(pred) or not len(gt): return np.empty((len(gt), len(pred)))
    ix1=np.maximum(pred[:,0:1].T,gt[:,0:1]); iy1=np.maximum(pred[:,1:2].T,gt[:,1:2])
    ix2=np.minimum(pred[:,2:3].T,gt[:,2:3]); iy2=np.minimum(pred[:,3:4].T,gt[:,3:4])
    inter=np.maximum(0,ix2-ix1)*np.maximum(0,iy2-iy1)
    ap=(pred[:,2]-pred[:,0])*(pred[:,3]-pred[:,1])
    ag=(gt[:,2]-gt[:,0])*(gt[:,3]-gt[:,1])
    u=ap[np.newaxis,:]+ag[:,np.newaxis]-inter
    return 1.0-np.where(u>0,inter/u,0.0)

def hota_approx(tp,fp,fn,ids):
    return float(np.sqrt(tp/max(tp+fp+fn,1)*max(0.0,1.0-ids/max(tp,1))))

def eval_acc(acc, name):
    s = mm.metrics.create().compute(acc,
        metrics=['mota','idf1','num_switches','recall','precision',
                 'num_misses','num_false_positives','num_matches'], name=name).iloc[0]
    return dict(mota=float(s['mota']), idf1=float(s['idf1']),
                recall=float(s['recall']), precision=float(s['precision']),
                ids=int(s['num_switches']), fn=int(s['num_misses']),
                fp=int(s['num_false_positives']), matches=int(s['num_matches']),
                hota=hota_approx(int(s['num_matches']),int(s['num_false_positives']),
                                 int(s['num_misses']),int(s['num_switches'])))

print('Helpers ready')

In [ ]:
# ── RUN DeepSORT ─────────────────────────────────────────────────
ts      = datetime.now().strftime('%Y%m%d_%H%M%S')
run_tag = f'FINAL_RUN4_DeepSORT_{ts}'

detector = YOLO(MODEL_NAME)
if HALF: detector.model.half()

rows = []

for seq in tqdm(VAL_SEQS, desc='DeepSORT'):
    gt = load_gt(ANNOT_DIR / f'{seq.name}.txt')
    if gt.empty: continue

    LOCAL_TMP.mkdir(exist_ok=True)
    ls = LOCAL_TMP / seq.name
    if ls.exists(): shutil.rmtree(ls)
    shutil.copytree(seq, ls)
    frames = sorted(ls.glob('*.jpg'))

    # Fresh DeepSORT instance per sequence
    tracker = DeepSort(
        max_age        = 30,
        n_init         = 3,
        nms_max_overlap= 1.0,
        max_cosine_dist= 0.3,
        nn_budget      = 100,
        embedder       = 'mobilenet',
        half           = True,
        bgr            = True,
    )

    acc   = mm.MOTAccumulator(auto_id=True)
    times = []

    for idx, fp in enumerate(frames, start=1):
        t0  = time.perf_counter()
        img = cv2.imread(str(fp))
        if img is None: continue

        # YOLOv8n detection (no tracking — DeepSORT handles tracking)
        det = detector(img, conf=0.25, iou=0.45, imgsz=640,
                       half=HALF, verbose=False, device=DEVICE)[0]

        # Convert detections to DeepSORT format: [[x1,y1,w,h], conf, class]
        ds_dets = []
        if det.boxes is not None and len(det.boxes):
            boxes_xyxy = det.boxes.xyxy.cpu().numpy()
            confs      = det.boxes.conf.cpu().numpy()
            for (x1,y1,x2,y2), c in zip(boxes_xyxy, confs):
                ds_dets.append(([x1, y1, x2-x1, y2-y1], float(c), 0))

        # Update DeepSORT
        tracks = tracker.update_tracks(ds_dets, frame=img)
        times.append(time.perf_counter() - t0)

        # Extract active confirmed tracks
        pred_ids   = []
        pred_boxes = []
        for t in tracks:
            if not t.is_confirmed(): continue
            ltrb = t.to_ltrb()
            pred_ids.append(t.track_id)
            pred_boxes.append(ltrb)

        pred_ids   = np.array(pred_ids,   dtype=int)
        pred_boxes = np.array(pred_boxes, dtype=float) if pred_boxes else np.empty((0,4))

        # GT for this frame
        gf = gt[gt['frame'] == idx]; gi = gf['id'].values
        gb = (np.column_stack([gf['x'].values, gf['y'].values,
                               gf['x'].values+gf['w'].values,
                               gf['y'].values+gf['h'].values])
              if len(gf) else np.empty((0,4)))

        dist = iou_dist(pred_boxes, gb)
        acc.update(gi, pred_ids, dist if dist.size else np.empty((len(gi), len(pred_ids))))

    shutil.rmtree(ls, ignore_errors=True)
    m   = eval_acc(acc, seq.name)
    fps = 1.0 / np.mean(times) if times else 0.0
    rows.append(dict(run_tag=run_tag, system='DeepSORT', sequence=seq.name,
                     frames=len(frames), fps=round(fps,2), **m))
    tqdm.write(f"DeepSORT {seq.name[:24]:24s} MOTA={m['mota']:.3f} "
               f"IDF1={m['idf1']:.3f} HOTA={m['hota']:.3f} "
               f"IDS={m['ids']:4d} FPS={fps:.1f}")

gc.collect()

df = pd.DataFrame(rows)
out = DRIVE_RESULTS / f'{run_tag}_per_seq.csv'
df.to_csv(out, index=False)

print(f'\n{"="*60}')
print(f"DeepSORT | MOTA={df['mota'].mean():.4f} IDF1={df['idf1'].mean():.4f} "
      f"HOTA={df['hota'].mean():.4f} IDS={int(df['ids'].sum())} FPS={df['fps'].mean():.1f}")
print(f'Saved -> {out.name}')